# 🔬 Notebook 3: Collaborative Whiteboard — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### LWW-map CRDT

We model the board as a **map of shape_id → shape** with a Last-Writer-Wins rule tied to a Lamport timestamp (plus actor id for tie-break).

Key property: applying the same set of ops in any order yields the same state. This is what makes collaborative editing converge.

In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Ts:
    lamport: int
    actor: str

class LWWBoard:
    def __init__(self):
        self.state = {}   # shape_id → (Ts, shape or None)

    def apply(self, op):
        ts = Ts(op["lamport"], op["actor"])
        sid = op["shape_id"] if op["kind"]=="delete" else op["shape"]["id"]
        cur = self.state.get(sid)
        if cur is None or ts > cur[0]:
            if op["kind"] == "delete":
                self.state[sid] = (ts, None)
            else:
                self.state[sid] = (ts, op["shape"])

    def shapes(self):
        return [s for _, s in self.state.values() if s is not None]

ops = [
    {"kind":"add","shape":{"id":"s1","x":0},"lamport":1,"actor":"alice"},
    {"kind":"update","shape":{"id":"s1","x":10},"lamport":2,"actor":"bob"},
    {"kind":"update","shape":{"id":"s1","x":5},"lamport":2,"actor":"alice"},  # same lamport → actor tie-break
]

# Apply in two different orders, confirm same result
for order in [ops, list(reversed(ops))]:
    board = LWWBoard()
    for o in order: board.apply(o)
    print("final:", board.shapes())

## Deep dive 2

### Fan-out with pub/sub

WebSocket servers are stateless; any user can connect to any node. To broadcast, each WS node subscribes to a Redis/NATS channel per room. The room's messages go through pub/sub so all nodes fan out to their local connections.

In [ ]:
# Toy pub/sub in-process
from collections import defaultdict

class Bus:
    def __init__(self):
        self.subs = defaultdict(list)
    def subscribe(self, topic, fn): self.subs[topic].append(fn)
    def publish(self, topic, msg):
        for fn in self.subs[topic]: fn(msg)

bus = Bus()
bus.subscribe("board:b1", lambda m: print("node-A got:", m))
bus.subscribe("board:b1", lambda m: print("node-B got:", m))
bus.publish("board:b1", {"kind":"add","id":"s1"})

## Closing thoughts

- CRDTs remove the need for a central serialization point → you can edit offline.
- Periodic **snapshots** keep the op log short.
- Presence (cursors) is *ephemeral* — don't persist it; pub/sub only.